# Imports of libraries and modules

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown

from langchain_community.document_loaders import JSONLoader, WebBaseLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_groq.chat_models import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.documents import Document

/var/folders/14/mdg60hc171q8ft30dvw15r7m0000gn/T/ipykernel_68471/787510328.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader, WebBaseLoader
/Users/vansh/Projects/arduchat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


# Constants and env files

In [2]:
load_dotenv()
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:
already_fetched_links = []

In [ ]:
question = "Give me paramerters related to potentail thrust loss"

# LLM and embeddings

In [5]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    reasoning_format="hidden",
)
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13907.65it/s]


# MAIN FUNCTIONS
### Getting json link file

In [6]:
json_data = JSONLoader(
    "./links_result_deduped.json",
    jq_schema=".[]",
    text_content=True
).load()
json_data

[Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 1}, page_content='https://ardupilot.org/copter/docs'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 2}, page_content='https://ardupilot.org/copter/docs/ArduCopter_MAVLink_Messages.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 3}, page_content='https://ardupilot.org/copter/docs/ac2_followme.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 4}, page_content='https://ardupilot.org/copter/docs/ac2_guidedmode.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 5}, page_content='https://ardupilot.org/copter/docs/ac2_positionmode.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 6}

### Making link vectordb and its retriever

In [7]:
vectorDB_link = Chroma.from_documents(
    documents=json_data,
    embedding=embedding
)
def retriever_link_generator():
    retriever_link = vectorDB_link.as_retriever()
    return retriever_link

In [8]:
link_retriever = retriever_link_generator()

In [9]:
link_retriever.invoke("circle")

[Document(metadata={'seq_num': 33, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/circle-mode.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 185}, page_content='https://ardupilot.org/copter/docs/common-compassless.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 834}, page_content='https://ardupilot.org/copter/docs/precision-landing-landmark.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 183}, page_content='https://ardupilot.org/copter/docs/common-compass-calibration-in-mission-planner.html')]

## Tag generation
### Prompt for tags

In [10]:
tags_prompt = PromptTemplate.from_template("""
You are a senior ArduPilot Copter engineer building search queries for a documentation retriever.

Given a junior engineer's question, extract 1-3 keywords/phrases that maximize retrieval accuracy against ArduPilot Copter docs.

Rules:
- Use exact ArduPilot terminology (parameter names like ATC_RAT_RLL_P, mode names like AUTO/LOITER/RTL, component names like EKF3, GPS, compass, ESC).
- Include the specific subsystem/feature (e.g. "geofence", "failsafe", "PID tuning", "motor output").
- Include synonyms only if they map to different doc sections (e.g. "compass calibration" AND "magnetometer calibration").
- Expand acronyms once if the full term aids retrieval (e.g. "EKF" -> also include "Extended Kalman Filter").
- Exclude generic filler words (drone, help, how, issue, problem).
- Order keywords from most specific to most general.
- Output ONLY a JSON array of strings. No explanation, no markdown, no preamble.

Question: {question}

Output:
"""
)

In [11]:
tags_prompt.invoke(question)

StringPromptValue(text='\nYou are a senior ArduPilot Copter engineer building search queries for a documentation retriever.\n\nGiven a junior engineer\'s question, extract 1-3 keywords/phrases that maximize retrieval accuracy against ArduPilot Copter docs.\n\nRules:\n- Use exact ArduPilot terminology (parameter names like ATC_RAT_RLL_P, mode names like AUTO/LOITER/RTL, component names like EKF3, GPS, compass, ESC).\n- Include the specific subsystem/feature (e.g. "geofence", "failsafe", "PID tuning", "motor output").\n- Include synonyms only if they map to different doc sections (e.g. "compass calibration" AND "magnetometer calibration").\n- Expand acronyms once if the full term aids retrieval (e.g. "EKF" -> also include "Extended Kalman Filter").\n- Exclude generic filler words (drone, help, how, issue, problem).\n- Order keywords from most specific to most general.\n- Output ONLY a JSON array of strings. No explanation, no markdown, no preamble.\n\nQuestion: Give me paramerters relate

### Making the chain for tags generation

In [12]:
tags_chain = tags_prompt | llm

In [13]:
x = tags_chain.invoke(question).content
x

'["THR_Loss", "THR_Detect", "FS_THR"]'

In [14]:
link_retriever.invoke(x)

[Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 31}, page_content='https://ardupilot.org/copter/docs/checklist.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 565}, page_content='https://ardupilot.org/copter/docs/common-rtk-correction.html'),
 Document(metadata={'seq_num': 717, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/crash_check.html'),
 Document(metadata={'seq_num': 262, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/common-f4by.html')]

### Retrieved question links

In [15]:
def weighted_query(tags):
    weighted = []
    for i, tag in enumerate(tags):
        weight = len(tags) - i  # first tag gets highest repeat count
        weighted.extend([tag] * weight)
    return " ".join(weighted)

In [16]:
def retrieved_question_links(question):
    tags_msg = tags_chain.invoke(question)
    if hasattr(tags_msg,"content"):
        tags = tags_msg.content
    print(tags)
    tags_string = weighted_query(tags)
    link_list = link_retriever.invoke(tags_string)
    return link_list

In [17]:
retrieved_question_links(question=question)

["AFTHR_LOSS_DISARM", "THR_LOSS_ALT", "AFTHR_CHK_TIMEOUT", "failsafe", "thrust loss disarming"]


[Document(metadata={'seq_num': 568, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/common-scripting-applets.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 31}, page_content='https://ardupilot.org/copter/docs/checklist.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 409}, page_content='https://ardupilot.org/copter/docs/common-mambaH743v4.html'),
 Document(metadata={'seq_num': 3, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/ac2_followme.html')]

In [18]:
def retrieved_question_links(question, decay=0.5):
    tags_msg = tags_chain.invoke(question)
    tags = eval(tags_msg.content) if hasattr(tags_msg, "content") else tags_msg
    # tags = ["Circle mode", "flight modes"]

    scored_docs = {}
    for i, tag in enumerate(tags):
        weight = decay ** i  # 1.0, 0.5, 0.25, ...
        results = link_retriever.vectorstore.similarity_search_with_score(tag, k=5)
        for doc, score in results:
            key = doc.page_content  # or doc.metadata['id']
            weighted_score = score * (1 / weight) if weight else score
            if key not in scored_docs or weighted_score < scored_docs[key][1]:
                scored_docs[key] = (doc, weighted_score)

    ranked = sorted(scored_docs.values(), key=lambda x: x[1])
    return [doc for doc, _ in ranked]

In [19]:
retrieved_links = retrieved_question_links(question=question)
retrieved_links

[Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 841}, page_content='https://ardupilot.org/copter/docs/radio-failsafe.html'),
 Document(metadata={'seq_num': 249, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/common-esc-issues.html'),
 Document(metadata={'seq_num': 717, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/crash_check.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 727}, page_content='https://ardupilot.org/copter/docs/failsafe-landing-page.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 128}, page_content='https://ardupilot.org/copter/docs/common-avanon-adc.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_d

### filtering whether the retrieved_links data is already present in the main_vector_db

In [20]:
def get_new_urls(retrieved_docs, already_fetched_links):
    seen = set(already_fetched_links)
    final_urls_for_data_loading = []
    for doc in retrieved_docs:
        url = doc.page_content
        if url and url not in seen:
            final_urls_for_data_loading.append(url)
            seen.add(url)
    return final_urls_for_data_loading

In [21]:
final_urls_for_data_loading = get_new_urls(retrieved_links, already_fetched_links)
already_fetched_links.extend(final_urls_for_data_loading)

In [22]:
retrieved_links

[Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 841}, page_content='https://ardupilot.org/copter/docs/radio-failsafe.html'),
 Document(metadata={'seq_num': 249, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/common-esc-issues.html'),
 Document(metadata={'seq_num': 717, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/crash_check.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 727}, page_content='https://ardupilot.org/copter/docs/failsafe-landing-page.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 128}, page_content='https://ardupilot.org/copter/docs/common-avanon-adc.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_d

In [23]:
print(final_urls_for_data_loading)
print(already_fetched_links)

['https://ardupilot.org/copter/docs/radio-failsafe.html', 'https://ardupilot.org/copter/docs/common-esc-issues.html', 'https://ardupilot.org/copter/docs/crash_check.html', 'https://ardupilot.org/copter/docs/failsafe-landing-page.html', 'https://ardupilot.org/copter/docs/common-avanon-adc.html', 'https://ardupilot.org/copter/docs/common-smart-battery-rotoye.html', 'https://ardupilot.org/copter/docs/solo_battery_calibration.html', 'https://ardupilot.org/copter/docs/failsafe-battery.html', 'https://ardupilot.org/copter/docs/common-tattu-dronecan-battery.html', 'https://ardupilot.org/copter/docs/common-aerotate-dronecan-battery.html', 'https://ardupilot.org/copter/docs/common-tkoff-rpm-min.html', 'https://ardupilot.org/copter/docs/common-minim-osd-quick-installation-guide.html', 'https://ardupilot.org/copter/docs/common-mambaf405-mini.html', 'https://ardupilot.org/copter/docs/common-tmotor-h7-mini.html', 'https://ardupilot.org/copter/docs/common-holybro-kakutef4-mini.html']
['https://ardup

### Getting docs from the final_urls_for_data_loading

In [24]:
loader = WebBaseLoader(final_urls_for_data_loading)
loader.requests_per_second = 10  # be polite to the server
docs = loader.load()    # concurrent fetch via aiohttp

In [25]:
doc_list = docs  # already List[Document] from loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_split = text_splitter.split_documents(doc_list)
print(doc_list[:2])


[Document(metadata={'source': 'https://ardupilot.org/copter/docs/radio-failsafe.html', 'title': 'Radio Failsafe — Copter  documentation', 'language': 'en'}, page_content='\n\n\n\n\nRadio Failsafe — Copter  documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nHome\n\nCopter\nPlane\nRover\nBlimp\nSub\nAntennaTracker\nMission Planner\nAPM Planner 2\nMAVProxy\nCompanion Computers\nDeveloper\n\n\nDownloads\n\nMission Planner\nAPM Planner 2\nAdvanced User Tools\nDeveloper Tools\nFirmware\n\n\nCommunity\n\nSupport Forums\nFacebook\nDeveloper Chat (Discord)\nDeveloper Voice (Discord)\nContact us\nGetting involved\nCommercial Support\nDevelopment Team\nUAS Training Centers\n\n\nStores\nAbout\n\nNews\nHistory\nLicense\nTrademark\nAcknowledgments\nWiki Editing Guide\nPartners Program\n\n\n\n\n\n\n\n\n\n            Copter\n              \n\n\n\n\n\n\n\n\n\n\nIntroduction to Copter\nChoosing an Autopilot\nGround Control Stations\nFirst Time Setup\nInstall Ground Station Software\nAutopilot Sy

In [26]:
vectorstore_main = Chroma.from_documents(
    documents=doc_split,
    embedding=embedding
)
main_retriever = vectorstore_main.as_retriever()

In [27]:
vectorstore_main.add_documents(documents=doc_split)

['30b7de06-2799-4de6-a3b9-d99bbfb5d0b5',
 '344fee9a-77db-442d-bbaf-14ca91b1c3f7',
 'a94204c8-e1c1-4b4c-888b-18f4a0c9e407',
 '7d653d86-84ae-4776-b7d4-69dad34b478b',
 'eeb97ab2-7ba8-43ff-9250-7904537d2c63',
 'c1f49d77-b9a8-4729-9ef7-097ab6ab9cfe',
 '1889920a-7b35-406b-a94d-4a68e2b0ed80',
 '7c22b304-a2fb-49c0-97e7-e42bddc83136',
 '72da1ade-9c5e-4c94-b84c-f052faa83bb7',
 'ae7c28a7-2af3-49c1-b895-06301fde7a30',
 'd9d39ef4-3338-40ec-bb2b-bb4cb594f147',
 '4c98e754-3a89-42ba-8fc2-ccf639f54788',
 'f7135283-7e18-441f-a382-2ecd948a3274',
 '39ddc450-07f0-44af-aa49-59bf597e3299',
 '197e25f4-6adf-4a1b-b4b4-d69821926f26',
 '9475b700-0b04-4b40-8d77-263c5971c7ad',
 '2768520a-0e76-459b-b5c7-787ad35d79fb',
 'a48b6c72-2d6d-4937-80ee-a06c23c28d25',
 '1e5a0a61-7004-470e-bf5f-ba16eb9171d7',
 '4f3070cf-b99a-458d-b31c-e7a635f7a85e',
 '61d4c141-cfcf-442a-8d27-65d3120a1c5f',
 '5a521aa1-493d-47a9-8da7-85e605300daa',
 '0a166f35-4bbc-4ed3-af0b-14359f4417c4',
 '5d5e9920-b64f-4e07-856d-14022c0d5f72',
 'b1cbd2b2-628b-

In [28]:
main_retriever.invoke(question)

[Document(metadata={'seq_num': 875, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/thrust_loss_yaw_imbalance.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 710}, page_content='https://ardupilot.org/copter/docs/common_baro_thrust_compensation.html'),
 Document(metadata={'seq_num': 754, 'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/motor-thrust-scaling.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/src/links_result_deduped.json', 'seq_num': 27}, page_content='https://ardupilot.org/copter/docs/booster-motor.html')]

### Retrival from main vectordb

In [29]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using the context below.\n\nContext:\n{context}. Links in the answer should strictly from {already_visited}. "
     "Any ArduPilot parameter you mention must come strictly from this list — do not invent or alter parameter names, "
     "and if none of these fit, don't mention a parameter at all:\n{valid_params}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])


In [30]:
chain = (
    {
        "context": lambda x: (x["question"]),
        "question": lambda x: x["question"],
        "history": lambda x: x["history"],   # <-- pass it through
        "already_visited": lambda x: list(retrieved_links),
    }
    | prompt2
    | llm
)

In [31]:
config = {"configurable": {"session_id": "user_1"}}

### parameter retriever

In [32]:
import requests
from bs4 import BeautifulSoup

URL = "https://ardupilot.org/copter/docs/parameters.html"

resp = requests.get(URL)
resp.encoding = "utf-8"  # fixes the Â¶ mojibake from wrong auto-detected encoding
soup = BeautifulSoup(resp.text, "html.parser")

h3_tags = [h3.get_text(strip=True).replace("¶", "").strip() for h3 in soup.find_all("h3")]

for tag in h3_tags:
    print(tag)

print(f"\nTotal: {len(h3_tags)}")

FORMAT_VERSION: Eeprom format version number
PILOT_THR_FILT: Throttle filter cutoff
PILOT_THR_BHV: Throttle stick behavior
GCS_PID_MASK: GCS PID tuning mask
RTL_CONE_SLOPE: RTL cone slope
RTL_LOIT_TIME: RTL loiter time
RTL_ALT_TYPE: RTL mode altitude type
FS_GCS_ENABLE: Ground Station Failsafe Enable
GPS_HDOP_GOOD: GPS Hdop Good
SUPER_SIMPLE: Super Simple Mode
WP_YAW_BEHAVIOR: Yaw behaviour during missions
FS_THR_ENABLE: Throttle Failsafe Enable
FS_THR_VALUE: Throttle Failsafe Value
THR_DZ: Throttle deadzone
FLTMODE1: Flight Mode 1
FLTMODE2: Flight Mode 2
FLTMODE3: Flight Mode 3
FLTMODE4: Flight Mode 4
FLTMODE5: Flight Mode 5
FLTMODE6: Flight Mode 6
FLTMODE_CH: Flightmode channel
INITIAL_MODE: Initial flight mode
SIMPLE: Simple mode bitmask
LOG_BITMASK: Log bitmask
ESC_CALIBRATION: ESC Calibration
TUNE: Tuning Parameter
FRAME_TYPE: Frame Type (+, X, V, etc)
DISARM_DELAY: Disarm delay
PHLD_BRK_RATE: PosHold braking rate
LAND_REPOSITION: Land repositioning
FS_EKF_ACTION: EKF Failsafe Act

In [33]:
param_docs = [Document(page_content=p) for p in h3_tags]
param_vectordb = Chroma.from_documents(param_docs, embedding=embedding)
param_retriever = param_vectordb.as_retriever(search_kwargs={"k": 15})

In [34]:
chain = (
    {
        "context": lambda x: main_retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "history": lambda x: x["history"],
        "already_visited": lambda x: list(retrieved_links),
        "valid_params": lambda x: param_retriever.invoke(x["question"]),
    }
    | prompt2
    | llm
)

In [35]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

/Users/vansh/Projects/arduchat/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [36]:
r2 = chain_with_history.invoke({"question": question}, config=config)
Markdown(r2.content)

Here are the parameters related to potential thrust loss from the provided context:

1. **`MOT_THST_HOVER`**  
   - **Description**: Thrust Hover Value. This parameter defines the motor thrust required to hover, which can be affected by factors like battery voltage or motor efficiency. Adjusting this value helps account for thrust loss during flight.  
   - **Documentation**: [Motor Thrust Scaling](https://ardupilot.org/copter/docs/motor-thrust-scaling.html)  

2. **`BARO1_THST_SCALE`**  
   - **Description**: Thrust compensation for barometric pressure changes. This parameter adjusts for thrust loss due to altitude or atmospheric variations.  
   - **Documentation**: [Common Baro Thrust Compensation](https://ardupilot.org/copter/docs/common_baro_thrust_compensation.html)  

3. **`SIM_SB_MOT_THST`**  
   - **Description**: Simulated motor thrust value (used in simulations). This can model potential thrust loss for testing purposes.  
   - **Documentation**: [Thrust Loss and Yaw Imbalance](https://ardupilot.org/copter/docs/thrust_loss_yaw_imbalance.html)  

These parameters directly address thrust-related adjustments and diagnostics. For further details, refer to the linked documents.